# 04_xgb_mono — ③ 단조 제약 XGBoost (xgb 담당)

`model_template`과 `02_logit`의 흐름을 그대로 따릅니다(4단계 부트스트랩 → 6a Dev 100% 재학습). 숫자는 모두 `contract/params.json`(00_config)에서 옵니다.

②와 **단조 제약만** 다릅니다(데이터·분할·파라미터·시드 동일). 제약 부호는 `lib_prep.FEATURE_SPEC`의 부호표에서 옵니다(`mono_constraints`).

**실행 방법: `런타임 → 모두 실행`.** 결과는 `outputs/xgb_mono/`(Colab이면 팀 Drive)에 저장됩니다.

- **먼저 시험**: 2절 `B_RUN = 3`으로 한 번 돌려 회차당 시간을 보고, 출력된 예상 시간을 팀에 알린 뒤 `B_RUN = B`로 본 실행합니다.
- **끊겨도 이어서**: 회차마다 `outputs/xgb_mono/rounds/`에 결과를 저장합니다. 다시 실행하면 같은 설정·같은 분할로 끝난 회차는 건너뜁니다.
- **나눠서 동시에**: 2절 `B_FROM, B_TO`를 사람마다 다르게(예: `0, 50`과 `50, 100`) 두고 같은 Drive 폴더에서 돌립니다. 모든 회차가 모이면 마지막 사람의 실행에서 5절(집계·최종 모형·summary)이 돌아갑니다. 회차가 빠져 있으면 5절은 빠진 회차 번호를 보여 주고 멈춥니다.
- **GPU**: Colab에서 `런타임 → 런타임 유형 변경 → GPU`면 자동으로 씁니다. **②와 ③, 그리고 나눠 돌린 회차 모두 같은 장치(GPU 또는 CPU)여야 합니다**(장치에 따라 결과가 아주 조금 다름). 섞이면 5절이 멈춥니다.
- `XGB_PARAMS`가 아직 미정(`PENDING`)이면 임시값으로 돕니다. 그 결과는 시험용이고, 확정 뒤 다시 돌려야 합니다(설정이 바뀌면 저장된 회차는 자동으로 다시 계산).


In [ ]:
import sys
if "google.colab" in sys.modules:
    !curl -sL https://raw.githubusercontent.com/Walnutbrown/kdt14th-lendingclub/main/colab_setup.py -o /content/colab_setup.py
    %run -i /content/colab_setup.py
# 1. 설정 불러오기
MODEL_KIND = "xgb_mono"        # 이 노트북에서 바꾸지 않음 (03_xgb = "xgb", 04_xgb_mono = "xgb_mono")
SKIP_LIB_TESTS = True
%run -i ./00_config.ipynb
%run -i ./lib_prep.ipynb
%run -i ./lib_engine.ipynb
for name in ("make_oob_splits", "split_fingerprint", "sr_curve", "pick_cutoff", "aggregate_bootstrap", "stack_curves",
             "fit_on", "assert_fit_on", "_idx_fingerprint", "make_preprocessor", "check_columns", "mono_constraints",
             "config_fingerprint", "run_meta", "models_b_manifest", "set_korean_font"):
    assert name in globals(), f"{name}가 없음 → 00_config·lib_prep·lib_engine이 최신 main인지 확인"


In [ ]:
# 2. 이번 실행 설정 (여기서 바꾸는 것은 이 세 줄뿐)
B_RUN = B                 # 본 실행 = 00_config의 B. 시험할 때만 3
B_FROM, B_TO = 0, B_RUN   # 나눠 돌릴 때 사람마다 다르게 (예: 0, 50 / 50, 100)
DEVICE = None             # None이면 GPU가 있으면 "cuda", 없으면 "cpu"

import os, time, shutil, subprocess, hashlib, joblib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, brier_score_loss
from xgboost import XGBClassifier

if DEVICE is None:
    has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    DEVICE = "cuda" if has_gpu else "cpu"
assert 0 <= B_FROM < B_TO <= B_RUN <= B, (B_FROM, B_TO, B_RUN, B)
OUT = OUT_DIR / MODEL_KIND
ROUND_DIR = OUT / "rounds"
for d in (OUT, OUT / "models_b", ROUND_DIR):
    d.mkdir(parents=True, exist_ok=True)
set_korean_font()
if "XGB_PARAMS" in PENDING:
    print("⚠️ XGB_PARAMS 미정(임시값) → 이 결과는 시험용입니다. 확정 뒤 다시 실행하세요.")
print(f"{MODEL_KIND} | B_RUN={B_RUN} 이번 회차 {B_FROM}~{B_TO - 1} | 장치 {DEVICE} | 설정해시[{MODEL_KIND}] {config_fingerprint(MODEL_KIND)}")
print("XGB 파라미터:", {**XGB_FIXED, **XGB_PARAMS})


In [ ]:
# 3. 데이터 (01_prep 결과). 행 순서를 바꾸지 않음
DATA_FILES = {n: hashlib.sha256((DATA_DIR / n).read_bytes()).hexdigest() for n in ("dev_X_raw.parquet", "dev_meta.parquet")}
X = pd.read_parquet(DATA_DIR / "dev_X_raw.parquet")
meta = pd.read_parquet(DATA_DIR / "dev_meta.parquet")
assert DATA_FILES == {n: hashlib.sha256((DATA_DIR / n).read_bytes()).hexdigest() for n in DATA_FILES}, "읽는 중에 Dev parquet이 바뀜 → 다시 실행"
assert len(X) == len(meta), "X와 meta의 행 수가 다름"
if len(X) != EXPECT["n_dev"]:
    print(f"⚠️ Dev 행 수 {len(X):,} ≠ EXPECT {EXPECT['n_dev']:,} → 01_prep 결과·parquet 버전 확인")
y = meta["y"].to_numpy().astype(int)
excess = meta["excess_ret"].to_numpy(dtype=float)
assert np.isfinite(excess).all(), "excess_ret에 결측/무한값이 있음"
CATS = load_categories()
splits = make_oob_splits(n_samples=len(X), n_bootstrap=B_RUN, seed=SEED)   # ①②③ 같은 분할
SPLIT_HASH = split_fingerprint(splits, len(X))
print("Dev", X.shape, f"부도율 {y.mean():.4f} | 분할 지문 split_hash = {SPLIT_HASH}")


In [ ]:
# 4. 회차별 학습 (Train_b에서만 전처리·모형 fit, OOB는 평가만. 회차마다 저장 → 끊겨도 이어서)
def fit_xgb(Xt, yt, prep):
    params = {**XGB_FIXED, **XGB_PARAMS, "device": DEVICE}   # OOB로 조기 종료하지 않음
    if MODEL_KIND == "xgb_mono":
        params["monotone_constraints"] = mono_constraints(prep)
    return XGBClassifier(**params).fit(Xt, yt)

def round_key(b, oob):
    """이 회차 결과를 다시 써도 되는지 판단하는 열쇠. 하나라도 다르면 다시 계산한다."""
    return dict(model=MODEL_KIND, b=b, oob=_idx_fingerprint(oob)[1], n=len(X), cfg=config_fingerprint(MODEL_KIND),
                lib_prep=LIB_PREP_HASH, data=DATA_FILES, device=DEVICE)

def round_paths(b):
    return ROUND_DIR / f"round_b{b:03d}.json", OUT / "models_b" / f"model_b{b:03d}.joblib"

def load_round(b, key):
    j, mf = round_paths(b)
    if not (j.exists() and mf.exists()):
        return None
    rec = json.loads(j.read_text(encoding="utf-8"))
    return rec if rec["key"] == key else None

def save_atomic(path, write):
    tmp = path.with_name(path.name + ".tmp")
    write(tmp)
    os.replace(tmp, path)   # 쓰는 도중 끊겨도 반쯤 쓴 파일이 남지 않게

if MODEL_KIND == "xgb_mono":
    _mono = mono_constraints(fit_on(make_preprocessor(MODEL_KIND, CATS), X, np.arange(min(len(X), 1000))))
    print(f"단조 제약: -1 {_mono.count(-1)}개, +1 {_mono.count(1)}개, 0 {_mono.count(0)}개 (설계: -1 9개, +1 11개)")

t_start, n_done = time.time(), 0
for b in range(B_FROM, B_TO):
    tr, oob = splits[b]
    key = round_key(b, oob)
    if load_round(b, key) is not None:
        print(f"b={b:3d} 이미 있음 → 건너뜀"); continue
    t0 = time.time()
    prep = fit_on(make_preprocessor(MODEL_KIND, CATS), X, tr)
    assert_fit_on(prep, tr)
    check_columns(prep, MODEL_KIND)
    model = fit_xgb(prep.transform(X.iloc[tr]), y[tr], prep)
    p = model.predict_proba(prep.transform(X.iloc[oob]))[:, 1]
    curve = sr_curve(p, excess[oob], RATES, SR_MODE)
    a, tau, sr = pick_cutoff(curve, MIN_RATE)
    row = dict(b=b, a_star=a, tau_star=tau, sr_max=sr, auc=roc_auc_score(y[oob], p), brier=brier_score_loss(y[oob], p))
    j, mf = round_paths(b)
    save_atomic(mf, lambda t: joblib.dump((prep, model), t))                     # 6-C 안정성용 (05_test가 읽음)
    rec = dict(key=key, row=row, curve=curve[["rate", "sr", "tau"]].to_dict("list"))
    save_atomic(j, lambda t: t.write_text(json.dumps(rec, default=float), encoding="utf-8"))   # 모형 다음에 씀 = 완료 표시
    n_done += 1
    per = (time.time() - t_start) / n_done
    print(f"b={b:3d} a*={a:.2f} SR*={sr:.4f} AUC={row['auc']:.4f} ({time.time() - t0:.0f}초) | 남은 {B_TO - b - 1}회 예상 {per * (B_TO - b - 1) / 60:.0f}분")
print(f"이번 실행 {n_done}회 학습, {(time.time() - t_start) / 60:.1f}분")


In [ ]:
# 5. 집계 (B_RUN회가 모두 모였을 때만) → 6a Dev 100% 재학습 → 저장
recs = {b: load_round(b, round_key(b, splits[b][1])) for b in range(B_RUN)}
missing = [b for b, r in recs.items() if r is None]
if missing:
    raise SystemExit(f"아직 없는(또는 설정·장치가 다른) 회차 {len(missing)}개: {missing[:30]}{' …' if len(missing) > 30 else ''} "
                     "→ 그 회차를 맡은 사람이 끝낸 뒤 이 노트북을 다시 실행하세요. (summary는 쓰지 않음)")
extra = sorted(p.name for p in (OUT / "models_b").glob("model_b*.joblib") if int(p.stem[7:]) >= B_RUN)
assert not extra, f"models_b에 B_RUN={B_RUN}보다 큰 회차 파일이 있음 {extra[:5]} → 이전 실행의 잔여물. 지우거나 B_RUN을 맞추세요"

results = [recs[b]["row"] for b in range(B_RUN)]
curves = [pd.DataFrame(recs[b]["curve"]) for b in range(B_RUN)]
res_df, summ = aggregate_bootstrap(results)
curves_df = stack_curves(curves)              # 계약 열 (b, rate, sr, tau)
print(res_df.drop(columns="b").describe().loc[["mean", "50%", "min", "max"]].round(4))

# 그림: 회차별 곡선과 중앙값 곡선 (실제 a*는 06_collect가 세 모형을 모아 중앙값 곡선 최대로 정함)
med = curves_df.groupby("rate")["sr"].median()
fig, ax = plt.subplots(figsize=(6.5, 3.2))
for _, g in curves_df.groupby("b"):
    ax.plot(g["rate"], g["sr"], color="0.85", lw=0.6)
ax.plot(med.index, med.values, color="C1", lw=2, label="회차 중앙값")
ax.axvline(float(med.idxmax()), color="C3", ls="--", lw=1, label=f"중앙값 곡선 최대 {med.idxmax():.2f}")
ax.set(xlabel="승인율", ylabel=f"OOB SR ({SR_MODE})", title=f"{MODEL_KIND}, B={B_RUN}")
ax.legend(frameon=False, fontsize=8); fig.tight_layout(); fig.savefig(OUT / "bootstrap_cutoff.png", dpi=300); plt.show()

t0 = time.time()
prep_full = fit_on(make_preprocessor(MODEL_KIND, CATS), X, np.arange(len(X)))
check_columns(prep_full, MODEL_KIND)
final = fit_xgb(prep_full.transform(X), y, prep_full)
joblib.dump((prep_full, final), OUT / "final_model.joblib")
print(f"Dev 100% 재학습 {time.time() - t0:.0f}초")

res_df.to_csv(OUT / "bootstrap_results.csv", index=False)
curves_df.to_csv(OUT / "bootstrap_curves.csv", index=False)
RESULT_FILES = {n: hashlib.sha256((OUT / n).read_bytes()).hexdigest() for n in ("bootstrap_results.csv", "bootstrap_curves.csv", "final_model.joblib")}
summary = dict(
    model=MODEL_KIND, a_star=summ["a_star"],   # 회차별 a*_b 중앙값 (06_collect·99_freeze 점검용 계약). 설계의 a*는 06_collect의 중앙값 곡선 최대
    tau_star_median=float(res_df["tau_star"].median()), a_star_cv=summ["a_star_cv"], tau_star_cv=summ["tau_star_cv"],
    more_stable=summ.get("more_stable"), sr_max_mean=summ["sr_max_mean"], n_failed=summ.get("n_failed", 0),
    B=B_RUN, B_config=B, seed=SEED, sr_mode=SR_MODE, min_rate=MIN_RATE,
    split_hash=SPLIT_HASH, n_samples=len(X), data_files=DATA_FILES, files=RESULT_FILES,
    models_b_files=models_b_manifest(OUT),
    hyperparameters={**XGB_FIXED, **XGB_PARAMS}, device=DEVICE,   # 99_freeze가 hyperparameters = XGB_FIXED + XGB_PARAMS인지 대조
    monotone_constraints=(list(mono_constraints(prep_full)) if MODEL_KIND == "xgb_mono" else None),
    columns=list(prep_full.get_feature_names_out()),
    config_hash=config_fingerprint(), lib_prep_hash=LIB_PREP_HASH, meta=run_meta())
(OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1, default=str), encoding="utf-8")   # 항상 마지막에 씀
print("저장:", OUT.relative_to(ROOT))
print(json.dumps({k: summary[k] for k in ("model", "B", "split_hash", "a_star", "device", "config_hash", "lib_prep_hash")}, ensure_ascii=False))
